# conda 환경설정

In [1]:
!conda create -n kmu-chatbot python=3.10 -y
!conda activate kmu-chatbot

Channels:
 - conda-forge
 - nvidia
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /home/20243161/.conda/envs/kmu-chatbot

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.5.20-hbd8a1cb_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_102 
  libexpat           conda-forge/linux-64::libexpat-2.8.1-hecca717_0 
  libffi             conda-forge/linux-64::libffi-3.5.2-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0feb66_19 
  libgcc-ng          conda-forge/linux-64::libgcc-ng-15.2.0-h69a702a_19 
  libgomp            conda-forge/linux-64::libgomp-15.2.0-he0feb66_19 
  liblzma            conda-forge/linux-64::liblzma-5.8.3-hb03c661_0 
  libnsl          

# 라이브러리 설치

In [2]:
!pip install -U pip

!pip install torch transformers accelerate peft bitsandbytes safetensors
!pip install sentence-transformers faiss-cpu numpy

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: http://repo.ai.gato/registry/repository/pypi-proxy/simple
  Using cached http://repo.ai.gato/registry/repository/pypi-proxy/packages/pip/26.1.2/pip-26.1.2-py3-none-any.whl (1.8 MB)

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: http://repo.ai.gato/registry/repository/pypi-proxy/simple

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: http://repo.ai.gato/registry/repository/pypi-proxy/simple

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


# 경로설정

In [3]:
from pathlib import Path
import os

PROJECT_DIR = Path("/home/20243161/kmu_chatbot_lora_package")

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

LORA_PATH = PROJECT_DIR / "lora" / "qwen3b_lora_clean" / "checkpoint-372"

VECTOR_DIR = PROJECT_DIR / "vector_db"
INDEX_PATH = VECTOR_DIR / "kmu_notice_index.faiss"
STORE_PATH = VECTOR_DIR / "kmu_notice_store.pkl"

print("PROJECT_DIR:", PROJECT_DIR)
print("PROJECT_DIR exists:", PROJECT_DIR.exists())

print("\nLoRA checkpoint:", LORA_PATH)
print("LoRA path exists:", LORA_PATH.exists())
print("adapter_config exists:", (LORA_PATH / "adapter_config.json").exists())
print("adapter_model exists:", (LORA_PATH / "adapter_model.safetensors").exists())

print("\nVector DB:")
print("FAISS index exists:", INDEX_PATH.exists())
print("Store pkl exists:", STORE_PATH.exists())

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"PROJECT_DIR not found: {PROJECT_DIR}")

if not (LORA_PATH / "adapter_config.json").exists():
    raise FileNotFoundError(f"adapter_config.json not found: {LORA_PATH}")

if not (LORA_PATH / "adapter_model.safetensors").exists():
    raise FileNotFoundError(f"adapter_model.safetensors not found: {LORA_PATH}")

if not INDEX_PATH.exists():
    raise FileNotFoundError(f"FAISS index not found: {INDEX_PATH}")

if not STORE_PATH.exists():
    raise FileNotFoundError(f"Store pkl not found: {STORE_PATH}")

print("\n경로 설정 완료")

PROJECT_DIR: /home/20243161/kmu_chatbot_lora_package
PROJECT_DIR exists: True

LoRA checkpoint: /home/20243161/kmu_chatbot_lora_package/lora/qwen3b_lora_clean/checkpoint-372
LoRA path exists: True
adapter_config exists: True
adapter_model exists: True

Vector DB:
FAISS index exists: True
Store pkl exists: True

경로 설정 완료


# 모델+LoRA 및 Vector DB 로드

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

USE_4BIT = True

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

print("Attaching LoRA checkpoint...")

model = PeftModel.from_pretrained(
    base_model,
    str(LORA_PATH)
)

model.eval()

def get_model_device():
    return next(model.parameters()).device

print("Qwen2.5-3B-Instruct + LoRA checkpoint 로드 완료")
print("model device:", get_model_device())

/usr/gatoai/python/venv/3.12/lib/python3.12/site-packages/numpy/_core/getlimits.py:559: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/gatoai/python/venv/3.12/lib/python3.12/site-packages/numpy/_core/getlimits.py:91: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/gatoai/python/venv/3.12/lib/python3.12/site-packages/numpy/_core/getlimits.py:559: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/gatoai/python/venv/3.12/lib/python3.12/site-packages/numpy/_core/getlimits.py:91: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Attaching LoRA checkpoint...
Qwen2.5-3B-Instruct + LoRA checkpoint 로드 완료
model device: cuda:0


In [5]:
import pickle
import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

VECTOR_DIR = PROJECT_DIR / "vector_db"

INDEX_PATH = VECTOR_DIR / "kmu_notice_index.faiss"
STORE_PATH = VECTOR_DIR / "kmu_notice_store.pkl"

print("INDEX_PATH:", INDEX_PATH)
print("STORE_PATH:", STORE_PATH)
print("FAISS exists:", INDEX_PATH.exists())
print("PKL exists:", STORE_PATH.exists())

if not INDEX_PATH.exists():
    raise FileNotFoundError(f"FAISS 파일 없음: {INDEX_PATH}")

if not STORE_PATH.exists():
    raise FileNotFoundError(f"PKL 파일 없음: {STORE_PATH}")

index = faiss.read_index(str(INDEX_PATH))

with open(STORE_PATH, "rb") as f:
    store = pickle.load(f)

print("store keys:", store.keys())
print("index ntotal:", index.ntotal)
print("documents:", len(store["documents"]))
print("metadatas:", len(store["metadatas"]))
print("embedding_model:", store.get("embedding_model"))
print("metric:", store.get("metric"))
print("normalized:", store.get("normalized"))

EMBED_MODEL = store.get("embedding_model", "intfloat/multilingual-e5-small")

embedder = SentenceTransformer(
    EMBED_MODEL,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Vector DB 로드 완료")

print("\n샘플 문서:")
print(store["documents"][0][:500])
print("\n샘플 metadata:")
print(store["metadatas"][0])

INDEX_PATH: /home/20243161/kmu_chatbot_lora_package/vector_db/kmu_notice_index.faiss
STORE_PATH: /home/20243161/kmu_chatbot_lora_package/vector_db/kmu_notice_store.pkl
FAISS exists: True
PKL exists: True
store keys: dict_keys(['ids', 'documents', 'embedding_docs', 'metadatas', 'bm25_corpus', 'embedding_model', 'metric', 'normalized', 'built_at'])
index ntotal: 2973
documents: 2973
metadatas: 2973
embedding_model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
metric: l2
normalized: False


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector DB 로드 완료

샘플 문서:
제목: 인공지능 연구실 (Machine Intelligence Lab.) 학부연구생 모집_6/14 23:00까지
출처: CS
분류: 학사공지
날짜: 26.05.28
◆ 지도교수 : 이재구연구실 홈페이지:
◆ 신청 마감: 6/14(일) 23:00
◆ 면담 일정: 6월 3주차 (향후, 개별 안내 예정)
◆ 연구실 관련 홍보 자료
1. 국민대학교, 생성AI 선도인재양성사업 지원대상 대학 선정
2. 국민대, ‘2025년도 AI스타펠로우십 지원사업’ 선정
3. 지도 교수 외부 초청 강연
◆ 관련 연구 주제
- 로봇/드론 Embodied AI/Physical AI 선행 연구 탐색
- 초거대 AI (LLM/LMM) 선행 연구 탐색
- AGI/초지능 선행 연구 탐색
- Deep Learning 포함한 AI/ML 기초
◆ 탐색대상 : 국민대 대학원 진학을 희망하는 소프트웨어 및 인공지능 학부 전공 혹은 복수전공 3/4학년 학생(방학 중 집중할 수 있는 학생 선발 예정)
◆ 학부 연구생 FAQ
- 무

샘플 metadata:
{'notice_id': 'd41d4d33edc9c46f', 'chunk_index': 0, 'chunk_id': 'd41d4d33edc9c46f_000', 'source': 'CS', 'category': '학사공지', 'title': '인공지능 연구실 (Machine Intelligence Lab.) 학부연구생 모집_6/14 23:00까지', 'date': '26.05.28', 'url': 'https://cs.kookmin.ac.kr/news/notice/2827', 'attachments': [], 'attached_links': ['https://www.hani.co.kr/arti/economy/biznews/1195590.html', 'https://buly.kr/8enaTWI', 'http://mi.kookmin.ac.kr/', 'https://www.donga.com/news/Society/artic

# 유틸리티 함수

In [6]:
from datetime import datetime
import re
import numpy as np
import torch


def parse_notice_date(date_value):
    if date_value is None:
        return None

    date_str = str(date_value).strip()
    nums = re.findall(r"\d+", date_str)

    if len(nums) < 3:
        return None

    year = int(nums[0])
    month = int(nums[1])
    day = int(nums[2])

    if year < 100:
        year += 2000

    try:
        return datetime(year, month, day)
    except Exception:
        return None


def recency_score(metadata: dict):
    notice_date = parse_notice_date(metadata.get("date"))

    if notice_date is None:
        return 0.0

    today = datetime.today()
    days_old = (today - notice_date).days

    if days_old <= 0:
        return 1.0
    if days_old <= 30:
        return 0.95
    if days_old <= 60:
        return 0.85
    if days_old <= 90:
        return 0.75
    if days_old <= 180:
        return 0.55
    if days_old <= 365:
        return 0.35

    return 0.15


def tokenize_simple(text: str):
    text = str(text).lower()
    return re.findall(r"[가-힣a-zA-Z0-9]+", text)


def is_general_course_registration_query(question: str):
    q = question.lower()

    if "수강신청" not in q and "수강 신청" not in q:
        return False

    special_terms = [
        "계절", "하계", "동계",
        "urop",
        "초과",
        "추가",
        "선도주제",
        "연구",
        "교과목",
        "대학원",
        "현장실습",
        "교환",
        "군",
        "폐강",
        "포기",
        "정정",
        "변경"
    ]

    return not any(term in q for term in special_terms)


def is_graduation_requirement_query(question: str):
    """
    졸업 요건/졸업 조건/졸업 기준/학번별 졸업요건을 묻는 질문인지 판단.
    """
    q = question.lower().replace(" ", "")

    if "졸업" not in q:
        return False

    requirement_terms = [
        "요건",
        "조건",
        "기준",
        "이수",
        "졸업요건",
        "졸업조건",
        "졸업기준",
        "학번",
        "교육과정"
    ]

    return any(term in q for term in requirement_terms)


def normalize_query(question: str):
    q = question.strip()

    if is_general_course_registration_query(q):
        return q + " 정규학기 학부 수강신청 일정 안내 기간 장바구니 방법"

    if is_graduation_requirement_query(q):
        return (
            q
            + " 연도별 학년도별 학번별 졸업요건 졸업 조건 졸업 기준 "
            + "전공별 이수요건 교육과정 졸업인증 2020 2021 2022 2023 2024 2025 2026"
        )

    if "장학" in q and not any(
        word in q for word in ["국가", "근로", "성적", "교내", "교외", "학자금", "등록금", "감면"]
    ):
        return q + " 장학공지 장학금 신청 기간 학생지원팀"

    if "등록금" in q and not any(
        word in q for word in ["분할", "환불", "고지서"]
    ):
        return q + " 등록금 납부 기간 고지서 확인"

    if "졸업" in q:
        return q + " 졸업 안내 졸업요건 졸업사정 졸업인증"

    return q

In [7]:
def title_based_score(question: str, search_question: str, document: str, metadata: dict):
    title = str(metadata.get("title", "")).lower()
    category = str(metadata.get("category", "")).lower()
    source = str(metadata.get("source", "")).lower()
    doc = str(document).lower()

    q = question.lower().strip()
    search_q = search_question.lower().strip()

    q_tokens = tokenize_simple(q)
    search_tokens = tokenize_simple(search_q)

    score = 0.0

    q_no_space = q.replace(" ", "")
    title_no_space = title.replace(" ", "")

    if q_no_space and q_no_space in title_no_space:
        score += 10.0

    for token in q_tokens:
        if len(token) <= 1:
            continue

        if token in title:
            score += 6.0

        if token in category:
            score += 1.5

        if token in source:
            score += 1.0

        if token in doc[:1500]:
            score += 0.8

    for token in search_tokens:
        if len(token) <= 1:
            continue

        if token in title:
            score += 2.0

        if token in doc[:1500]:
            score += 0.4

    # 기본 공지 선호
    basic_terms = [
        "안내",
        "모집",
        "신청 안내",
        "일정 안내",
        "수강신청 일정",
        "등록금 납부",
        "장학금 신청",
        "졸업요건",
        "졸업 요건",
        "학사"
    ]

    for term in basic_terms:
        if term in title:
            score += 1.5

    # 수강신청 기본 공지 우선
    if is_general_course_registration_query(question):
        if "수강신청 일정 안내" in title:
            score += 25.0

        if "수강신청 일정" in title:
            score += 20.0

        if "일정 안내" in title and "수강신청" in title:
            score += 15.0

        if "장바구니" in title or "장바구니" in doc:
            score += 3.0

        negative_terms = [
            "초과수강",
            "초과 수강",
            "추가 수강신청",
            "추가수강신청",
            "선도주제연구",
            "교과목 수강신청",
            "urop",
            "계절학기",
            "하계",
            "동계",
            "현장실습",
            "대학원",
            "폐강",
            "수강포기",
            "수강 정정",
            "수강정정"
        ]

        for term in negative_terms:
            if term in title:
                score -= 25.0

    # 졸업 요건 질문 보정
    if is_graduation_requirement_query(question):
        # 원하는 문서
        if "졸업요건" in title or "졸업 요건" in title:
            score += 35.0

        if "졸업조건" in title or "졸업 조건" in title:
            score += 28.0

        if "졸업기준" in title or "졸업 기준" in title:
            score += 28.0

        if "학년도별" in title or "년도별" in title or "연도별" in title:
            score += 28.0

        if "학번별" in title or "학번" in title:
            score += 25.0

        if "교육과정" in title:
            score += 18.0

        if "이수요건" in title or "이수 요건" in title:
            score += 18.0

        if re.search(r"20\d{2}", title) and ("졸업" in title or "요건" in title):
            score += 20.0

        # 원하지 않는 문서
        negative_terms = [
            "초과학기",
            "초과 학기",
            "초과수강",
            "졸업탈락",
            "탈락 위험",
            "학위증",
            "학위수여",
            "졸업식",
            "수료",
            "유예",
            "졸업대상자",
            "졸업 사정 결과",
            "전공필수 미이수"
        ]

        for term in negative_terms:
            if term in title or term in doc[:500]:
                score -= 35.0

    elif "졸업" in question:
        if "졸업" in title:
            score += 15.0
        if "학위" in title:
            score += 5.0

    # 장학
    if "장학" in question:
        if "장학" in title:
            score += 12.0
        if "장학금" in title:
            score += 15.0

    # 등록금
    if "등록금" in question:
        if "등록금" in title:
            score += 15.0
        if "납부" in title:
            score += 8.0

    return score

In [8]:
# =========================
# 최신 공지 질문 처리 패치
# 이 셀 하나만 맨 아래에 추가 실행하면 됨
# =========================
from datetime import datetime


def is_latest_notice_query(question: str):
    """
    사용자가 '가장 최근 공지', '최신 공지', '최근 올라온 공지'를 요구하는지 판단
    """
    q = question.lower().replace(" ", "")

    latest_terms = [
        "최근공지",
        "최신공지",
        "가장최근공지",
        "최근올라온공지",
        "새로운공지",
        "새공지",
        "최근소식",
        "최신소식"
    ]

    if any(term in q for term in latest_terms):
        return True

    if "공지" in q and any(term in q for term in ["최근", "최신", "새로운", "새로", "방금", "올라온", "가장최근"]):
        return True

    return False


def retrieve_latest_notices(top_k: int = 3, debug: bool = True):
    """
    검색어 의미 검색 없이, 날짜 기준 최신 공지 top_k 반환
    """
    results = []

    for idx, document in enumerate(store["documents"]):
        metadata = store["metadatas"][idx]

        notice_date = parse_notice_date(metadata.get("date"))

        results.append({
            "idx": idx,
            "score": 1.0,
            "relevance_score": 1.0,
            "title_score": 0.0,
            "recency_score": recency_score(metadata),
            "vector_score": 0.0,
            "notice_date": notice_date,
            "document": document,
            "metadata": metadata
        })

    # 날짜 있는 공지 우선 + 최신순
    results = sorted(
        results,
        key=lambda x: (
            x["notice_date"] is not None,
            x["notice_date"] or datetime.min
        ),
        reverse=True
    )

    # 같은 notice_id 중복 제거
    deduped = []
    seen_notice_ids = set()

    for item in results:
        metadata = item["metadata"]
        notice_id = metadata.get("notice_id", item["idx"])

        if notice_id in seen_notice_ids:
            continue

        seen_notice_ids.add(notice_id)
        deduped.append(item)

        if len(deduped) >= top_k:
            break

    if debug:
        print("\n===== 최신 공지 결과 =====")
        for i, item in enumerate(deduped, 1):
            m = item["metadata"]
            print(
                i,
                m.get("title"),
                "| date:", m.get("date"),
                "| url:", m.get("url")
            )

    return deduped

In [9]:
def collect_notice_chunks(notice_id, max_chunks: int = 8):
    chunks = []

    for doc, meta in zip(store["documents"], store["metadatas"]):
        if meta.get("notice_id") == notice_id:
            chunks.append({
                "document": doc,
                "metadata": meta
            })

    chunks = sorted(
        chunks,
        key=lambda x: x["metadata"].get("chunk_index", 0)
    )

    return chunks[:max_chunks]


def build_context_from_results(results, max_doc_chars_per_notice: int = 3500):
    context_parts = []

    for i, item in enumerate(results, 1):
        metadata = item["metadata"]

        notice_id = metadata.get("notice_id", item["idx"])

        title = metadata.get("title", "제목 없음")
        category = metadata.get("category", "분류 없음")
        date = metadata.get("date", "날짜 없음")
        source = metadata.get("source", "출처 없음")
        url = metadata.get("url", "")

        chunks = collect_notice_chunks(notice_id)

        full_text_parts = []

        if chunks:
            for chunk in chunks:
                full_text_parts.append(str(chunk["document"]))
        else:
            full_text_parts.append(str(item["document"]))

        full_text = "\n\n".join(full_text_parts)

        context_parts.append(
            f"[문서 {i}]\n"
            f"제목: {title}\n"
            f"분류: {category}\n"
            f"날짜: {date}\n"
            f"출처: {source}\n"
            f"URL: {url}\n"
            f"내용:\n{full_text[:max_doc_chars_per_notice]}"
        )

    return "\n\n---\n\n".join(context_parts)


def build_reference_notice_block(results):
    if len(results) == 0:
        return ""

    lines = []
    lines.append("\n\n---\n")
    lines.append("참고 공지")

    for i, item in enumerate(results, 1):
        metadata = item["metadata"]

        title = metadata.get("title", "제목 없음")
        date = metadata.get("date", "날짜 없음")
        url = metadata.get("url", "")

        lines.append(f"\n{i}. {title}")
        lines.append(f"   - 날짜: {date}")

        if url:
            lines.append(f"   - URL: {url}")
        else:
            lines.append("   - URL: 제공된 공지 데이터에서 확인되지 않습니다.")

    return "\n".join(lines)


def is_detail_poor_context(context: str):
    """
    공지 본문이 이미지/첨부 중심이라 텍스트 정보가 부족한지 판단.
    """
    text = str(context).strip()

    # 너무 짧으면 세부 내용 부족으로 판단
    if len(text) < 1200:
        return True

    # 내용은 있는데 실질적인 안내 키워드가 적으면 부족하다고 판단
    detail_keywords = [
        "신청", "기간", "대상", "방법", "제출", "서류", "문의",
        "일정", "장소", "운영", "활동", "모집", "선발", "유의",
        "학점", "등록", "수강", "졸업", "요건"
    ]

    count = sum(1 for kw in detail_keywords if kw in text)

    if count <= 2:
        return True

    return False


def has_relevant_notice(question: str, retrieved: list):
    """
    검색된 공지가 질문과 어느 정도 관련 있는지 판단.
    제목 기반으로 판단.
    """
    if len(retrieved) == 0:
        return False

    q_tokens = tokenize_simple(question)

    for item in retrieved:
        metadata = item["metadata"]
        title = str(metadata.get("title", "")).lower()
        document = str(item.get("document", "")).lower()

        # 검색 점수가 어느 정도 있으면 관련 있다고 판단
        if item.get("title_score", 0) >= 5:
            return True

        for token in q_tokens:
            if len(token) <= 1:
                continue

            if token in title:
                return True

            # 제목에는 없지만 본문 앞부분에 있으면 관련 가능성 있음
            if token in document[:1000]:
                return True

    return False

In [10]:
# =========================
# 제목 매칭 기반 답변 분기 패치
# =========================

def title_matches_question(question: str, item: dict):
    """
    질문 키워드가 공지 제목에 들어있는지 판단.
    제목에 있으면 관련 공지로 인정.
    """
    metadata = item["metadata"]
    title = str(metadata.get("title", "")).lower()

    q = str(question).lower().strip()
    q_no_space = q.replace(" ", "")
    title_no_space = title.replace(" ", "")

    # 질문 전체가 제목에 들어가면 강한 매칭
    if q_no_space and q_no_space in title_no_space:
        return True

    q_tokens = tokenize_simple(question)

    meaningful_tokens = [
        token for token in q_tokens
        if len(token) >= 2 and token not in ["알려줘", "뭐야", "어디", "확인", "공지", "관련"]
    ]

    if not meaningful_tokens:
        return False

    # 질문의 의미 있는 토큰 중 하나라도 제목에 있으면 관련 공지로 인정
    for token in meaningful_tokens:
        if token in title:
            return True

    return False


def has_title_matched_notice(question: str, retrieved: list):
    """
    검색 결과 중 제목이 질문과 직접 매칭되는 공지가 있는지 확인.
    """
    for item in retrieved:
        if title_matches_question(question, item):
            return True
    return False


def is_text_detail_enough(context: str):
    """
    본문 텍스트에 실제 안내 정보가 충분한지 판단.
    이미지/첨부 중심 공지는 텍스트가 짧거나 키워드가 부족할 수 있음.
    """
    text = str(context).strip()

    # 너무 짧으면 부족
    if len(text) < 1500:
        return False

    detail_keywords = [
        "신청", "기간", "대상", "방법", "제출", "서류", "문의",
        "일정", "장소", "운영", "활동", "모집", "선발", "유의",
        "학점", "등록", "수강", "졸업", "요건", "자격", "혜택"
    ]

    count = sum(1 for kw in detail_keywords if kw in text)

    # 안내 키워드가 충분히 있으면 본문이 있다고 판단
    return count >= 3


def build_title_only_notice_answer(question: str, retrieved: list):
    """
    제목에는 관련 공지가 있으나 본문 세부 텍스트가 부족할 때의 답변.
    """
    main = retrieved[0]
    metadata = main["metadata"]

    title = metadata.get("title", "관련 공지")
    date = metadata.get("date", "날짜 없음")
    url = metadata.get("url", "")

    answer = (
        f"'{title}' 공지가 확인되었습니다. "
        f"해당 공지는 {date}에 올라온 공지이며, 질문하신 내용은 이 공지에서 확인하는 것이 좋습니다.\n\n"
        "다만 현재 벡터 DB에 저장된 본문 텍스트만으로는 세부 내용이 충분히 추출되지 않았습니다. "
        "공지 내용이 이미지, 표, 첨부파일 형태로 포함되어 있을 수 있으므로 자세한 일정, 대상, 신청 방법, 유의사항은 아래 참고 공지에서 직접 확인해주세요."
    )

    if url:
        answer += f"\n\n확인 URL: {url}"

    return answer


def build_no_notice_answer(question: str):
    """
    제목에도 매칭되는 공지가 없을 때의 답변.
    """
    return (
        f"'{question}'와 직접적으로 관련된 공지는 찾지 못했습니다. "
        "제공된 공지 데이터의 제목 기준으로는 해당 내용을 확인하기 어렵습니다. "
        "키워드를 조금 더 구체적으로 입력하거나 국민대학교 공지사항에서 직접 확인해주세요."
    )

In [11]:
def is_absence_only_answer(answer: str):
    """
    Qwen이 관련 공지가 있는데도 '없다' 식으로만 답했는지 판단.
    """
    answer = str(answer).strip()

    absence_phrases = [
        "공지에 명시되어 있지 않습니다",
        "공지에서 확인되지 않습니다",
        "제공된 공지에서는 확인되지 않습니다",
        "확인되지 않습니다",
        "확인할 수 없습니다",
        "나와 있지 않습니다",
        "명시되어 있지 않습니다"
    ]

    # 답변이 너무 짧고, 부정 문구가 있으면 fallback 대상
    if len(answer) < 250 and any(p in answer for p in absence_phrases):
        return True

    # 부정 문구만 반복하는 경우도 fallback
    if any(p in answer for p in absence_phrases) and not any(
        key in answer for key in ["신청", "기간", "대상", "방법", "문의", "URL", "날짜", "공지명"]
    ):
        return True

    return False


def build_found_notice_fallback_answer(question: str, retrieved: list):
    """
    제목에는 관련 공지가 있는데 Qwen이 없다고 답한 경우 사용할 fallback 답변.
    """
    main = retrieved[0]
    metadata = main["metadata"]

    title = metadata.get("title", "관련 공지")
    date = metadata.get("date", "날짜 없음")
    url = metadata.get("url", "")

    answer = (
        f"'{question}'와 관련된 공지는 확인되었습니다. "
        f"가장 관련 있는 공지는 '{title}'이며, 게시일은 {date}입니다.\n\n"
        "다만 현재 벡터 DB에 저장된 본문 텍스트만으로는 세부 내용을 충분히 추출하지 못했습니다. "
        "공지 본문이 이미지, 표, 첨부파일 형태로 포함되어 있거나 텍스트 추출이 일부 누락되었을 수 있습니다.\n\n"
        "따라서 자세한 신청 기간, 대상, 신청 방법, 제출 서류, 유의사항 등은 아래 참고 공지에서 직접 확인해주세요."
    )

    if url:
        answer += f"\n\n확인 URL: {url}"

    return answer

# 공지 검색 및 응답 생성 함수

In [12]:
def retrieve_notices_final(
    question: str, top_k: int = 3, candidate_k: int = 40, faiss_k: int = 80, debug: bool = True
):
    """최신 공지 질문이면 날짜순 검색, 아니면 FAISS + Title 일반 검색 수행"""
    # 1. 최신 공지 분기
    if is_latest_notice_query(question):
        if debug:
            print("💡 최신 공지 요청으로 판단됨:", question)
        return retrieve_latest_notices(top_k=top_k, debug=debug)

    # 2. 일반 검색 분기 (기존 _retrieve_notices_final_base 내용 통합)
    search_question = normalize_query(question)

    if debug:
        print("원본 질문:", question)
        print("검색용 질문:", search_question)

    query_embedding = np.array(embedder.encode([f"query: {search_question}"], normalize_embeddings=True), dtype="float32")
    faiss_scores, faiss_indices = index.search(query_embedding, faiss_k)
    faiss_score_map = {int(idx): float(score) for score, idx in zip(faiss_scores[0], faiss_indices[0]) if idx != -1}
    title_score_map = {}    

    for idx, document in enumerate(store["documents"]):
        score = title_based_score(question, search_question, document, store["metadatas"][idx])
        if score > 0: title_score_map[idx] = score

    title_top_indices = sorted(title_score_map.keys(), key=lambda i: title_score_map[i], reverse=True)[:candidate_k]
    candidate_indices = set(faiss_score_map.keys()) | set(title_top_indices)
    max_title = max(title_score_map.values()) if title_score_map else 1.0
    max_faiss = max(faiss_score_map.values()) if faiss_score_map else 1.0
    
    chunk_results = []

    for idx in candidate_indices:
        meta = store["metadatas"][idx]
        t_score = title_score_map.get(idx, 0.0)
        f_score = faiss_score_map.get(idx, 0.0)
        r_score = recency_score(meta)

        final_score = ((t_score / max_title) * 0.70) + (r_score * 0.20) + ((f_score / max_faiss) * 0.10)
        
        chunk_results.append({
            "idx": idx, "score": final_score, "title_score": t_score, 
            "recency_score": r_score, "vector_score": f_score, 
            "document": store["documents"][idx], "metadata": meta
        })

    chunk_results = sorted(chunk_results, key=lambda x: (x["score"], x["recency_score"]), reverse=True)
    notice_results, seen = [], set()

    for item in chunk_results:
        notice_id = item["metadata"].get("notice_id", item["idx"])
        if notice_id not in seen:
            seen.add(notice_id)
            notice_results.append(item)
            if len(notice_results) >= top_k: break
                
    if debug:
        print("\\n===== 최종 검색 결과 =====")
        for i, item in enumerate(notice_results, 1):
            print(i, item["metadata"].get("title"), "| final:", round(item["score"], 4))
            
    return notice_results


def generate_answer_final(
    question: str, top_k: int = 3, candidate_k: int = 40, debug: bool = False, show_references: bool = False
):
    """4개의 패치가 하나로 융합된 최종 답변 생성 함수 (반환형식: answer, ref)"""
    # 검색된 공지 리스트(ref)를 받아옵니다.
    retrieved = retrieve_notices_final(question, top_k, candidate_k, debug=debug)

    # 1. 아예 검색된게 없을 때
    if len(retrieved) == 0:
        return build_no_notice_answer(question), retrieved

    # 💡 1. 최신 공지 여부 확인
    is_latest = is_latest_notice_query(question)
    
    # 💡 2. 최신 공지 질문이면 제목 일치 여부를 무조건 True로 프리패스!
    title_matched = True if is_latest else has_title_matched_notice(question, retrieved)
    
    context = build_context_from_results(retrieved, max_doc_chars_per_notice=5000)
    detail_enough = is_text_detail_enough(context)

    if debug:
        print("제목 매칭 여부:", title_matched)
        print("본문 상세 텍스트 충분 여부:", detail_enough)

    # 2. 제목 매칭이 아예 안될 때
    if not title_matched:
        return build_no_notice_answer(question), retrieved

    # 3. 제목은 매칭되는데, 이미지 위주라서 텍스트가 부족할 때 (바로 Fallback 안내)
    # 단, 최신 공지를 묻는 거라면 LLM이 일단 제목과 날짜라도 요약하도록 패스합니다.
    if title_matched and not detail_enough and not is_latest:
        answer = build_found_notice_fallback_answer(question, retrieved)
        if show_references: 
            answer += build_reference_notice_block(retrieved)
        return answer, retrieved

    # 4. 정보가 충분하여 Qwen 모델을 통해 답변 생성
    # 💡 최신 공지와 일반 질문의 프롬프트를 다르게 줍니다.
    if is_latest:
        sys_msg = "너는 국민대학교 공지사항 안내 챗봇이다. 제공된 최신 공지 목록을 바탕으로 사용자에게 최근 소식들을 친절하게 요약해서 안내해라."
        user_msg = (
            f"사용자 질문: {question}\n\n최근 공지 본문:\n{context}\n\n"
            "위 공지들을 최신순으로 간단히 소개하고, 각각 어떤 내용인지 핵심만 2~3줄로 요약해줘. "
            "없는 정보는 절대 지어내지 말고, '답변 지침' 같은 프롬프트 문구는 출력하지 마."
        )
    else:
        sys_msg = (
            "너는 국민대학교 공지사항 안내 챗봇이다. 반드시 제공된 공지 본문만 근거로 답변한다. "
            "없는 정보를 지어내면 안 되며, 확인되는 내용 중심으로 자연스럽고 친절하게 답변한다. "
        )
        user_msg = (
            f"사용자 질문: {question}\n\n"
            f"검색된 공지 본문:\n{context}\n\n"
            "너는 제공된 공지 본문을 읽고 다음 [처리 순서]에 따라 답변해야 해.\n\n"
            "[처리 순서]\n"
            "1. 관련성 평가: 검색된 공지가 사용자의 질문(예: 시험 날짜)을 해결할 수 있는 본질적인 내용인지 판단해. (예: '간식 행사', '야식 배부' 등은 시험 일정과 무관하므로 부적합함)\n"
            "2. 분기 처리:\n"
            "   - 만약 본문이 질문과 무관하거나 정보가 부족하다면 억지로 지어내지 말고, '제공된 공지는 OOO에 대한 내용이므로, 질문하신 내용에 대한 정확한 정보는 없습니다.'라고만 답변해.\n"
            "   - 만약 본문이 질문과 일치한다면, 일정, 대상, 방법을 꼼꼼하게 요약해줘.\n\n"
            "주의: '처리 순서', '관련성 평가' 같은 너의 사고 과정이나 프롬프트 지시문은 최종 결과에 절대 출력하지 마."
        )

    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192)
    inputs = {k: v.to(get_model_device()) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=1500, do_sample=False, repetition_penalty=1.04, pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

    # 프롬프트 지시문 누수 방지
    for phrase in ["답변 지침", "관련 공지 본문", "사용자에게 답변", "프롬프트", "내부 지시문"]:
        answer = answer.replace(phrase, "")

    # 5. 일반 질문인데 모델이 멍청하게 "없다"고만 답변했을 때를 위한 안전장치 (Fallback)
    if not is_latest and title_matched and is_absence_only_answer(answer):
        if "대한 내용이며" not in answer and "다른 내용" not in answer:
            answer = build_found_notice_fallback_answer(question, retrieved)

    # (옵션) 기존처럼 텍스트 끝에 참고 공지 목록을 붙일지 여부
    if show_references:
        answer += build_reference_notice_block(retrieved)

    # 💡 최종 반환 (답변 텍스트, 참조 공지 메타데이터 리스트)
    return answer, retrieved

# 응답 Example

In [13]:
question = "최근 공지?"

# answer와 ref로 각각 나눠서 받습니다.
answer, ref = generate_answer_final(
    question, 
    top_k=3, 
    candidate_k=40, 
    debug=False, 
    show_references=False  # 프론트에서 ref를 따로 처리할 거라면 False로 두는 것이 좋습니다.
)

print("===== 챗봇 답변 =====")
print(answer)

print("\n===== 참조 문서 (ref) 메타데이터 확인 =====")
for i, notice in enumerate(ref, 1):
    meta = notice['metadata']
    print(f"[{i}] 제목: {meta.get('title')}")
    print(f"    URL: {meta.get('url')}")
    print(f"    검색점수: {round(notice.get('score', 0), 4)}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


===== 챗봇 답변 =====
공지 1: 인공지능 연구실 (Machine Intelligence Lab.) 학부연구생 모집이 진행되고 있습니다. 해당 연구실은 지도교수 이재구 연구실과 함께 다양한 AI 관련 연구를 수행하고 있습니다. 신청 마감일은 6월 14일(일) 23시까지이며, 면담 일정은 추후에 안내될 예정입니다.

공지 2: CJ올리브네트웍스에서 AI Campus 1기 교육생을 모집하고 있습니다. 이는 방학 중 집중할 수 있는 학생들을 위한 Stanford CS231N과 서울대 Deep Learning 등 기초 전공 스터디 프로그램입니다.

공지 3: [AI 네트워크 응용분야 아이디어 공모전] K-디지털 챌린지 : 넷 챌린지 캠프 시즌13이 열립니다. 이는 AI 네트워크 응용 분야의 아이디어 공모전으로, 다양한 아이디어를 제출할 수 있습니다.

===== 참조 문서 (ref) 메타데이터 확인 =====
[1] 제목: 인공지능 연구실 (Machine Intelligence Lab.) 학부연구생 모집_6/14 23:00까지
    URL: https://cs.kookmin.ac.kr/news/notice/2827
    검색점수: 1.0
[2] 제목: [CJ올리브네트웍스] AI Campus 1기 교육생을 모집
    URL: https://cs.kookmin.ac.kr/news/jobs/2084
    검색점수: 1.0
[3] 제목: [AI 네트워크 응용분야 아이디어 공모전] K-디지털 챌린지 : 넷 챌린지 캠프 시즌13
    URL: https://cs.kookmin.ac.kr/news/event/1859
    검색점수: 1.0


# API

## 환경설정

In [14]:
!pip install fastapi uvicorn pydantic nest-asyncio requests

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: http://repo.ai.gato/registry/repository/pypi-proxy/simple

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [15]:
import nest_asyncio
import uvicorn
import threading
import asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

In [16]:
nest_asyncio.apply()

## 스키마 정의

In [17]:
# 1. Pydantic 스키마 정의 (요청/응답 형식)
class ChatRequest(BaseModel):
    question: str
    top_k: int = 3
    candidate_k: int = 40

class ReferenceMeta(BaseModel):
    title: str
    category: str
    source: str
    date: str
    url: str
    score: float

class ChatResponse(BaseModel):
    answer: str
    references: list[ReferenceMeta]

## FastAPI 앱 생성

In [18]:
app = FastAPI(title="국민대 공지사항 챗봇 API (Notebook Ver.)")

# 💡 외부 프론트엔드/백엔드에서 접속할 수 있도록 CORS 문열어주기
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # 지금은 프로토타입이니 모든 웹사이트에서 접속 허용 ("*")
    allow_credentials=True,
    allow_methods=["*"],  # GET, POST 등 모든 통신 방식 허용
    allow_headers=["*"],  # 모든 헤더 허용
)

## API 엔드포인트 정의

In [19]:
@app.post("/api/v1/chat", response_model=ChatResponse)
async def chat_endpoint(request: ChatRequest):
    # 이미 위쪽 셀에서 메모리에 로드해둔 generate_answer_final 함수를 그대로 사용!
    answer, retrieved = generate_answer_final(
        question=request.question,
        top_k=request.top_k,
        candidate_k=request.candidate_k,
        debug=False,
        show_references=False
    )
    
    # 프론트엔드로 보낼 참조 데이터 가공
    formatted_refs = []
    for item in retrieved:
        meta = item["metadata"]
        formatted_refs.append(
            ReferenceMeta(
                title=meta.get("title", "제목 없음"),
                category=meta.get("category", "분류 없음"),
                source=meta.get("source", "출처 없음"),
                date=meta.get("date", "날짜 없음"),
                url=meta.get("url", ""),
                score=item.get("score", 0.0)
            )
        )
        
    return ChatResponse(answer=answer, references=formatted_refs)

## 백그라운드 스레드에서 서버 실행 함수

In [20]:
def run_server():
    # 💡 새로운 스레드를 위한 비동기 이벤트 루프 생성 (서버 안 튕기게 하는 핵심!)
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    print("🚀 API 서버 시작 중... (http://127.0.0.1:8080)")
    # 포트를 8080으로 변경
    config = uvicorn.Config(app, host="0.0.0.0", port=8080, log_level="info")
    server = uvicorn.Server(config)
    server.run()

## 스레드 실행

In [21]:
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

🚀 API 서버 시작 중... (http://127.0.0.1:8080)


### API 테스트 셀

In [22]:
import requests
import json
import time

# 서버가 안전하게 켜지도록 2초만 대기
time.sleep(2)

# 💡 포트를 8080으로 변경
url = "http://127.0.0.1:8080/api/v1/chat"

payload = {
    "question": "최근 공지?",
    "top_k": 3
}

print(f"[{payload['question']}] -> API로 질문을 전송하는 중...\n")

try:
    response = requests.post(url, json=payload)

    if response.status_code == 200:
        data = response.json()
        print("🤖 챗봇 답변:")
        print(data["answer"])
        print("\n🔗 참고 공지사항 데이터:")
        print(json.dumps(data["references"], indent=2, ensure_ascii=False))
    else:
        print("🚨 API 내부 오류:", response.status_code, response.text)
        
except requests.exceptions.ConnectionError:
    print("🚨 여전히 연결이 거부됩니다. 서버 스레드가 정상적으로 시작되지 않은 것 같습니다.")

INFO:     Started server process [17456]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)


[최근 공지?] -> API로 질문을 전송하는 중...

INFO:     127.0.0.1:50120 - "POST /api/v1/chat HTTP/1.1" 200 OK
🤖 챗봇 답변:
공지 1: 인공지능 연구실 (Machine Intelligence Lab.) 학부연구생 모집이 진행되고 있습니다. 해당 연구실은 지도교수 이재구 연구실과 함께 다양한 AI 관련 연구를 수행하고 있습니다. 신청 마감일은 6월 14일(일) 23시까지이며, 면담 일정은 추후에 안내될 예정입니다.

공지 2: CJ올리브네트웍스에서 AI Campus 1기 교육생을 모집하고 있습니다. 이는 방학 중 집중할 수 있는 학생들을 위한 Stanford CS231N과 서울대 Deep Learning 등 기초 전공 스터디 프로그램입니다.

공지 3: [AI 네트워크 응용분야 아이디어 공모전] K-디지털 챌린지 : 넷 챌린지 캠프 시즌13이 열립니다. 이는 AI 네트워크 응용 분야의 아이디어 공모전으로, 다양한 아이디어를 제출할 수 있습니다.

🔗 참고 공지사항 데이터:
[
  {
    "title": "인공지능 연구실 (Machine Intelligence Lab.) 학부연구생 모집_6/14 23:00까지",
    "category": "학사공지",
    "source": "CS",
    "date": "26.05.28",
    "url": "https://cs.kookmin.ac.kr/news/notice/2827",
    "score": 1.0
  },
  {
    "title": "[CJ올리브네트웍스] AI Campus 1기 교육생을 모집",
    "category": "취업공지",
    "source": "CS",
    "date": "26.05.28",
    "url": "https://cs.kookmin.ac.kr/news/jobs/2084",
    "score": 1.0
  },
  {
    "title": "[AI 네트워크 응용분야 아이디어 공모전] K-디지털

In [ ]:
# Cloudflare Tunnel 다운로드 및 실행 (경고창 없는 깔끔한 터널!)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8080

2026-06-02T10:37:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-02T10:37:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-02T10:37:24Z INF +--------------------------------------------------------------------------------------------+
2026-06-02T10:37:24Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-02T10:37:24Z INF |  https://professor-especially-spokesman-package.tryclo